In [ ]:
! pip install torch_geometric

In [ ]:
import requests
import os
import json
import pandas as pd
import gzip, json
from collections import defaultdict, Counter
from torch_geometric.data import Data

In [ ]:
urls = {"meta":"https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_Toys_and_Games.json.gz","reviews":"https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Toys_and_Games.json.gz"}
for name, url in urls.items():
  file_name = f"{name}_Toys_and_Games.json.gz"
  if not os.path.exists(file_name):
    print(f"Downloading {file_name}...")
    response = requests.get(url)
    with open(file_name, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"Done!")
  else:
    print(f"Already exists: {file_name}")


In [ ]:
def parse_gz(path):
    with gzip.open(path, "rb") as f:
        for line in f:
            try:
                yield json.loads(line)
            except:
                try:
                    yield eval(line)
                except:
                    pass

# Count labels at each depth level
by_depth = defaultdict(Counter)

for item in parse_gz("meta_Toys_and_Games.json.gz"):
    for path in item.get("categories", []):
        for depth, label in enumerate(path):
            by_depth[depth][label] += 1

for depth in sorted(by_depth.keys()):
    print(f"\n Level {depth} ({len(by_depth[depth])} unique labels) ")
    for label, count in by_depth[depth].most_common(20):
        print(f"  {count:>6}  {label}")

In [ ]:
records = []

for item in parse_gz("meta_Toys_and_Games.json.gz"):
    records.append(item)

df = pd.DataFrame(records)
print(df.shape)
print(df.columns.tolist())

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df['description'].notna().sum()

In [ ]:
both_missing = df['title'].isna() & df['description'].isna()
print(both_missing.sum())

In [ ]:
import torch
from torch_geometric.data import Data

# map every asin to an integer index
asin2idx = {asin: i for i, asin in enumerate(df['asin'])}

# build edge list
src_list = []
dst_list = []

edge_types = ['also_bought']

for _, row in df.iterrows():
    related = row['related']
    if not isinstance(related, dict):
        continue
    for etype in edge_types:
        for neighbor_asin in related.get(etype, []):
            if neighbor_asin in asin2idx:
                src_list.append(asin2idx[row['asin']])
                dst_list.append(asin2idx[neighbor_asin])

# add both directions
src_all = src_list + dst_list
dst_all = dst_list + src_list

#  remove duplicates
edges = set()
src_clean = []
dst_clean = []

for s, d in zip(src_all, dst_all):
    if s == d:
        continue
    if (s, d) in edges:
        continue
    edges.add((s, d))
    src_clean.append(s)
    dst_clean.append(d)


edge_index = torch.tensor([src_clean, dst_clean], dtype=torch.long)

print(f"Nodes:          {len(asin2idx)}")
print(f"Edges:          {edge_index.shape[1]}")

degree = Counter()
for src in edge_index[0].tolist():
    degree[src] += 1
print(f"Isolated nodes: {len(asin2idx) - len(degree)}")
print(f"Connected nodes: {len(degree)}")

In [ ]:
def get_label(categories):
    if not categories:
        return None
    for path in categories:
        if path and path[0] == 'Toys & Games' and len(path) >= 2:
            return path[1]
    return None  

df['label'] = df['categories'].apply(get_label)

df = df[df['label'].notna()].reset_index(drop=True)

print(f"Products after dropping level-0 only: {len(df)}")
print(f"Unique labels: {df['label'].nunique()}")
print()
print(df['label'].value_counts())

In [ ]:
label_counts = df['label'].value_counts()

print(f"Classes with >= 100 products: {(label_counts >= 100).sum()}")
print(f"Classes with >= 50 products:  {(label_counts >= 50).sum()}")
print(f"Classes with >= 10 products:  {(label_counts >= 10).sum()}")
print(f"Classes with < 10 products:   {(label_counts < 10).sum()}")
print(f"Classes with 1 product:       {(label_counts == 1).sum()}")
print()
print(f"Products in classes >= 100:   {label_counts[label_counts >= 100].sum()}")
print(f"Products in classes < 10:     {label_counts[label_counts < 10].sum()}")

In [ ]:
# encode labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label_idx'] = le.fit_transform(df['label'])

print(f"Classes: {df['label_idx'].nunique()}")
print(df[['label', 'label_idx']].drop_duplicates().sort_values('label_idx'))

In [ ]:
# rebuild asin2idx on df (no filtering needed, all 317k products kept)
asin2idx = {asin: i for i, asin in enumerate(df['asin'])}

src_list, dst_list = [], []
edge_types = ['also_bought']  

for _, row in df.iterrows():
    related = row['related']
    if not isinstance(related, dict):
        continue
    for etype in edge_types:
        for neighbor_asin in related.get(etype, []):
            if neighbor_asin in asin2idx:
                src_list.append(asin2idx[row['asin']])
                dst_list.append(asin2idx[neighbor_asin])

src_all = src_list + dst_list
dst_all = dst_list + src_list

edges = set()
src_clean, dst_clean = [], []
for s, d in zip(src_all, dst_all):
    if s == d: continue
    if (s, d) in edges: continue
    edges.add((s, d))
    src_clean.append(s)
    dst_clean.append(d)

edge_index = torch.tensor([src_clean, dst_clean], dtype=torch.long)
print(f"Nodes: {len(asin2idx)}")
print(f"Edges: {edge_index.shape[1]}")

# check isolation
degree = Counter()
for src in edge_index[0].tolist():
    degree[src] += 1
print(f"Isolated nodes: {len(asin2idx) - len(degree)}")
print(f"Connected nodes: {len(degree)}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import numpy as np

# combine title and description
df['text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
df['text'] = df['text'].str.strip()

# TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_tfidf = vectorizer.fit_transform(df['text'])

# reduce to 128 dims with SVD (same as PCA but works on sparse matrices)
svd = TruncatedSVD(n_components=128, random_state=42)
X_bow = svd.fit_transform(X_tfidf)

print(f"Feature matrix shape: {X_bow.shape}")
print(f"Explained variance:   {svd.explained_variance_ratio_.sum():.2%}")

In [ ]:
x = torch.tensor(X_bow, dtype=torch.float)
y = torch.tensor(df['label_idx'].values, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)
print(data)

In [ ]:
from sklearn.model_selection import train_test_split

idx = torch.arange(len(df))

# first split off test set
idx_train_val, idx_test = train_test_split(
    idx, test_size=0.15, random_state=42,
    stratify=df['label_idx']
)

# then split train and val
idx_train, idx_val = train_test_split(
    idx_train_val, test_size=0.15/0.85, random_state=42,
    stratify=df.loc[idx_train_val.numpy(), 'label_idx']
)

data.train_mask = torch.zeros(len(df), dtype=torch.bool)
data.val_mask   = torch.zeros(len(df), dtype=torch.bool)
data.test_mask  = torch.zeros(len(df), dtype=torch.bool)

data.train_mask[idx_train] = True
data.val_mask[idx_val]     = True
data.test_mask[idx_test]   = True

print(f"Train: {data.train_mask.sum()} ({data.train_mask.sum()/len(df):.0%})")
print(f"Val:   {data.val_mask.sum()} ({data.val_mask.sum()/len(df):.0%})")
print(f"Test:  {data.test_mask.sum()} ({data.test_mask.sum()/len(df):.0%})")

In [ ]:
print(data)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x):
        x = self.lin1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        return x

model = MLP(
    in_channels=128,      # BoW feature size
    hidden_channels=256,  # hidden layer size
    out_channels=142      # number of classes
)
print(model)

In [ ]:
from torch.optim import Adam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

data = data.to(device)
model = model.to(device)

optimizer = Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate():
    model.eval()
    out = model(data.x)
    pred = out.argmax(dim=1)

    train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean()
    val_acc   = (pred[data.val_mask]   == data.y[data.val_mask]).float().mean()
    test_acc  = (pred[data.test_mask]  == data.y[data.test_mask]).float().mean()

    return train_acc.item(), val_acc.item(), test_acc.item()

# training loop
best_val_acc = 0
best_test_acc = 0

for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = evaluate()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc

    if epoch % 20 == 0:
        print(f"Epoch {epoch:>3} | Loss: {loss:.4f} | "
              f"Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")

print(f"\nBest Val Acc:  {best_val_acc:.4f}")
print(f"Best Test Acc: {best_test_acc:.4f}")

In [ ]:
from torch_geometric.nn import SAGEConv

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model_sage = GraphSAGE(128, 256, 142).to(device)
optimizer_sage = Adam(model_sage.parameters(), lr=0.01, weight_decay=5e-4)

def train_sage():
    model_sage.train()
    optimizer_sage.zero_grad()
    out = model_sage(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer_sage.step()
    return loss.item()

@torch.no_grad()
def evaluate_sage():
    model_sage.eval()
    out = model_sage(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean()
    val_acc   = (pred[data.val_mask]   == data.y[data.val_mask]).float().mean()
    test_acc  = (pred[data.test_mask]  == data.y[data.test_mask]).float().mean()
    return train_acc.item(), val_acc.item(), test_acc.item()

best_val_acc = 0
best_test_acc = 0

for epoch in range(1, 201):
    loss = train_sage()
    train_acc, val_acc, test_acc = evaluate_sage()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc
    if epoch % 20 == 0:
        print(f"Epoch {epoch:>3} | Loss: {loss:.4f} | "
              f"Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")

print(f"\nBest Val Acc:  {best_val_acc:.4f}")
print(f"Best Test Acc: {best_test_acc:.4f}")

In [ ]:
!pip install sentence-transformers

### BERT

In [ ]:
from sentence_transformers import SentenceTransformer

model_bert = SentenceTransformer('all-MiniLM-L6-v2')  

texts = df['text'].tolist()
X_bert = model_bert.encode(texts, batch_size=256, show_progress_bar=True)

print(f"BERT embeddings shape: {X_bert.shape}")

In [ ]:
from sentence_transformers import SentenceTransformer

model_bert_large = SentenceTransformer('all-mpnet-base-v2')
texts = df['text'].tolist()
X_bert_large = model_bert_large.encode(texts, batch_size=128, show_progress_bar=True)

print(f"Shape: {X_bert_large.shape}")  

In [ ]:
import matplotlib.pyplot as plt

def train_sage_full(x_features, in_channels, model_name="GraphSAGE",
                    epochs=300, patience=20, lr=0.01):


    data.x = torch.tensor(x_features, dtype=torch.float).to(device)

    # model
    model = GraphSAGE(in_channels, 256, 142).to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    history = {'train': [], 'val': [], 'test': [], 'loss': []}
    best_val_acc = 0
    best_test_acc = 0
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        # train
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # evaluate
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc   = (pred[data.val_mask]   == data.y[data.val_mask]).float().mean().item()
            test_acc  = (pred[data.test_mask]  == data.y[data.test_mask]).float().mean().item()

        history['train'].append(train_acc)
        history['val'].append(val_acc)
        history['test'].append(test_acc)
        history['loss'].append(loss.item())

        # early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_test_acc = test_acc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        if epoch % 20 == 0:
            print(f"Epoch {epoch:>3} | Loss: {loss:.4f} | "
                  f"Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")

    print(f"\nBest Val Acc:  {best_val_acc:.4f}")
    print(f"Best Test Acc: {best_test_acc:.4f}")

    # plot
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history['train'], label='Train')
    plt.plot(history['val'], label='Val')
    plt.plot(history['test'], label='Test')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title(f'{model_name} - Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='Loss', color='red')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'{model_name} - Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    return best_val_acc, best_test_acc

In [ ]:
# BoW baseline
print(" GraphSAGE + BoW ")
val_bow, test_bow = train_sage_full(X_bow, in_channels=128, model_name="SAGE+BoW")

# BERT small 
print(" GraphSAGE + BERT MiniLM ")
val_minilm, test_minilm = train_sage_full(X_bert, in_channels=384, model_name="SAGE+MiniLM")



### CLIP

In [ ]:
from transformers import CLIPTokenizer, CLIPTextModel
import torch
import numpy as np


clip_text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

texts = df['text'].tolist()
X_clip_text = []

clip_text_model.eval()
batch_size = 256

with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = clip_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=77,
            return_tensors="pt"
        ).to(device)
        outputs = clip_text_model(**inputs)
        # extract pooled output → the [CLS] token embedding
        embeddings = outputs.pooler_output  # ← this is the tensor
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        X_clip_text.append(embeddings.cpu().numpy())
        if i % 10000 == 0:
            print(f"Processed {i}/{len(texts)}")

X_clip_text = np.vstack(X_clip_text)
print(f"CLIP text embeddings shape: {X_clip_text.shape}")

In [ ]:
print(" GraphSAGE + CLIP text ")
val_clip, test_clip = train_sage_full(
    X_clip_text,
    in_channels=512,
    model_name="SAGE+CLIP-text"
)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

def get_qwen_embeddings(model_name, texts, batch_size=64):
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
    model.eval()

    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)
            outputs = model(**inputs)
            # mean pooling
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            emb = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            embeddings.append(emb.cpu().float().numpy())
            if i % 5000 == 0:
                print(f"  {i}/{len(texts)}")

    return np.vstack(embeddings)

texts = df['text'].tolist()
X_qwen_06b = get_qwen_embeddings("Qwen/Qwen3-Embedding-0.6B", texts)
print(f"Shape: {X_qwen_06b.shape}")